In [ ]:
# This notebook will produce a feature table with mapped MS2 spectra and export the selected MS2 spectra in MSP format. 
# This will allow your data to be analyzed using GNPS or other MS2-based tools.

<a href="https://colab.research.google.com/github/shuzhao-li-lab/asari_pcpfm_tutorials/blob/main/tutorial/part_4/4.1_gnps_mixed_ms1ms2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Library imports for data processing

import pymzml
import os
import tqdm
import ftplib
import os
import intervaltree
import matchms
import pandas as pd

In [ ]:
# block for downloading the data

FTP_URL  = "massive-ftp.ucsd.edu"
FTP_PATH = "/v04/MSV000090156/peak/mzml/POS_MSMS/Lab_2/"
DATA_DIR = "./GNPS_testing/"

def download_dir(ftp, remote_dir, local_dir):
    os.makedirs(local_dir, exist_ok=False)
    ftp.cwd(remote_dir)

    for name in ftp.nlst():
        try:
            # try to CWD → it's a directory
            ftp.cwd(name)
            ftp.cwd("..")
            download_dir(ftp, remote_dir + "/" + name, local_dir + "/" + name)
        except ftplib.error_perm:
            # it's a file
            local_fp = os.path.join(local_dir, name)
            with open(local_fp, "wb") as f:
                ftp.retrbinary(f"RETR {name}", f.write)
                print("✔", local_fp)


with ftplib.FTP(FTP_URL) as ftp:
    ftp.login()  # anonymous OK
    download_dir(ftp, FTP_PATH, DATA_DIR)



In [ ]:
# Run asari for MS1 feature extraction

os.system(f"/Users/mitchjo/Library/Python/3.11/bin/asari process -i {DATA_DIR} -o {DATA_DIR} -m pos")

In [ ]:
# Set paths for dataset and find all MS2 spectra

all_mzml_files = []
for f in os.listdir(DATA_DIR):
    if f.lower().endswith("mzml"):
        all_mzml_files.append(os.path.join(DATA_DIR, f))

spectra = []
for mzml in tqdm.tqdm(all_mzml_files):
    for spec in pymzml.run.Reader(mzml):
        if spec.ms_level == 2:
            spec_rtime = spec.scan_time_in_minutes()*60
            for precursor in spec.selected_precursors:
                entry = {"spectrum": spec,
                         "rtime": spec_rtime,
                         "sample_origin": os.path.basename(mzml).rstrip(".mzML"),
                         "intensity_sum": sum(spec.i)}
                entry.update(precursor)
                spectra.append(entry)

print(f"Extracted {len(spectra)} MS2 Spectra")

In [ ]:
# find and read the feature table from the Asari run
for x in sorted(os.listdir(".")):
    if 'asari' in x:
        asari_dir = x
        break

PREF_FEATURE_TABLE_PATH = os.path.join(os.path.abspath("."), asari_dir, "preferred_Feature_table.tsv")
FULL_FEATURE_TABLE_PATH = os.path.join(os.path.abspath("."), asari_dir, "export/full_Feature_table.tsv")

print(f"Found preferred table at: {PREF_FEATURE_TABLE_PATH}")
print(f"Found full table at: {FULL_FEATURE_TABLE_PATH}")


In [ ]:
# this associated MS2 spectra to features
# the mapping is based on ppm and rt tolerance, then the most intense spectrum is selected of all matches. 
# this will output new tables co-located with the input tables and mgf files for upload to GNPS

def map_features_to_ms2(features, ms2_spectra, ppm_tol=5.0, rt_tol=10.0):
    spectrum_mz = intervaltree.IntervalTree()
    spectrum_rt = intervaltree.IntervalTree()
    id_to_spectrum = {}

    for s in ms2_spectra:
        mz_err = s['mz'] / 1e6 * ppm_tol
        key = (s['sample_origin'], s['spectrum'].ID)
        id_to_spectrum[key] = s
        spectrum_mz.addi(s['mz'] - mz_err, s['mz'] + mz_err, key)
        spectrum_rt.addi(s['rtime'] - rt_tol, s['rtime'] + rt_tol, key)

    out = []
    for f in features:
        mz_hits = {x.data for x in spectrum_mz.at(f['mz'])}
        rt_hits = {x.data for x in spectrum_rt.at(f['rtime'])}
        f['matches'] = mz_hits & rt_hits if mz_hits and rt_hits else set()
        out.append(f)

    return out, id_to_spectrum


def process_feature_table(
    feature_table_path,
    ms2_spectra,
):
    ft = pd.read_csv(feature_table_path, sep="\t")
    features = list(ft.to_dict(orient='records'))
    sample_cols = ft.columns[11:]
    non_sample_cols = ft.columns[:11]

    mapped, id_to_spectrum = map_features_to_ms2(features, ms2_spectra)

    features_w_ms2 = []
    spectra_to_export = []

    id_number_to_scan = {}

    for f in mapped:
        if not f['matches']:
            f['selected_ms2'] = ''
            features_w_ms2.append(f)
            continue

        possibles = [key for key in f['matches'] if f[key[0]] > 0]

        if possibles:
            scored = []
            for (sample_origin, spec_id) in possibles:
                spec = id_to_spectrum[(sample_origin, spec_id)]
                rt_err = abs(spec['rtime'] - f['rtime'])
                mz_err = abs(spec['mz'] - f['mz'])
                scored.append((rt_err, mz_err, spec,
                               round(spec['mz'], 4),
                               round(spec['rtime'], 4),
                               sample_origin))
            sel = sorted(scored, key=lambda x: x[1])[0]
            f['selected_ms2'] = f"{sel[3]}_{sel[4]}_{sel[5]}"
            try:
                sp = sel[2]['spectrum']
                spec_out = matchms.Spectrum(
                    sp.mz,
                    sp.i,
                    metadata={
                        "TITLE": f['id_number'][1:],
                        "PEPMASS": "0.0",
                        "CHARGE": "1",
                        "SCANS": f"{len(spectra_to_export) + 1}",
                        "COLLISION_ENERGY": "0.0",
                    }
                )
                spec_out = matchms.filtering.default_filters(spec_out)
                spec_out = matchms.filtering.normalize_intensities(spec_out)
                spectra_to_export.append(sp)
                id_number_to_scan[f['id_number']] = len(spectra_to_export)
            except:
                pass
        else:
            f['selected_ms2'] = ''
        del f['matches']
        features_w_ms2.append(f)

    out_msp_path = feature_table_path.replace(".tsv", "_ms2_spectra.mgf")
    with open(out_msp_path, "w") as fh:
        for i, sp in enumerate(spectra_to_export, start=1):
            fh.write("BEGIN IONS\n")
            fh.write(f"SCANS={i}\n")
            fh.write("PEPMASS=0.0\n")
            fh.write("CHARGE=1\n")
            fh.write("COLLISION_ENERGY=0.0\n")
            for m, inten in zip(sp.mz, sp.i):
                fh.write(f"{m} {inten}\n")
            fh.write("END IONS\n\n")

    print(f"Wrote {out_msp_path} with {i} MS2 Spectra")

    df = pd.DataFrame(features_w_ms2)
    new_df = pd.DataFrame()
    for x in non_sample_cols:
        new_df[x] = df[x]
    new_df['selected_ms2'] = df['selected_ms2']
    for x in sample_cols:
        new_df[x] = df[x]
    out_feature_table_path = feature_table_path.replace(".tsv", "_w_MS2.tsv")
    new_df.to_csv(out_feature_table_path, sep="\t", index=False)

    for_GNPS = pd.DataFrame()
    for_GNPS["row ID"]  = [id_number_to_scan.get(x, 0) for x in new_df["id_number"]]
    for_GNPS["row m/z"] = new_df['mz']
    for_GNPS['row retention time'] = new_df['rtime']
    for z in new_df.columns[12:]:
        for_GNPS[z + ' Peak area'] = new_df[z]
    for_GNPS = for_GNPS[for_GNPS["row ID"] != 0]
    for_GNPS.to_csv(feature_table_path.replace(".tsv", "_for_GNPS.csv"), index=False)
    print(f"Wrote {feature_table_path.replace(".tsv", "_for_GNPS.csv")} with {for_GNPS.shape[0]} Features")
    print(f"{round(i/for_GNPS.shape[0] * 100, 2)} percent of features have MS2")
    return for_GNPS


In [ ]:
# process the preferred table into GNPS
process_feature_table(PREF_FEATURE_TABLE_PATH, spectra)

# process the full table into GNPS
process_feature_table(FULL_FEATURE_TABLE_PATH, spectra)